# Genetic ancestry estimation from VCF files

Here we are going to learn how to estimate genetic ancestry from a VCF file. We need to follow these steps:

1. Download PLINK 2
2. Download the data
3. Convert to PLINK binary files
4. Perform quality control
5. Principal component analysis (PCA)
6. Download ADMIXTURE
7. Run ADMIXTURE
8. Plot the ADMIXTURE results

After each step, a table lists the files it creates, what they are and whether they are used later. A summary in Spanish, with a diagram of how the files connect, is in [archivos_por_paso.md](https://github.com/YTZ27/curso-ancestria/blob/main/notebooks/archivos_por_paso.md).

## 1. Download PLINK 2

If the link stops working, go to the [PLINK 2 page](https://www.cog-genomics.org/plink/2.0/), copy the **Linux 64-bit Intel** link from the **Alpha** column and paste it in the `wget` line. The rest of the cell stays the same.

In [ ]:
!wget -nv -O plink2.zip https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20261015.zip
!unzip -o plink2.zip
!./plink2 --version
!ls -lh plink2.zip plink2

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `plink2.zip` | The compressed download | No: it is only needed to extract `plink2` |
| `plink2` | The PLINK 2 program | Yes: in steps 3, 4 and 5 |

## 2. Download the data

The file `exome.vcf.gz` is stored in the course repository (folder `data/`). It contains individuals from the 1000 Genomes Project and the Human Genome Diversity Panel (HGDP), restricted to coding regions: the original VCF files ([PLINK 2 resources](https://www.cog-genomics.org/plink/2.0/resources)) were filtered with the BED file of the exome kit [Agilent SureSelectXT Human All Exon V8](https://www.agilent.com/en/product/next-generation-sequencing/ngs-assays-panels/exome/sureselect-human-all-exon-4252439).

In [ ]:
!wget -nv -nc https://github.com/YTZ27/curso-ancestria/raw/main/data/exome.vcf.gz
!ls -lh exome.vcf.gz

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `exome.vcf.gz` | Genotypes of 969 individuals at 244,305 variants, in VCF format (a text table), compressed with gzip (`.gz`) | Yes: it is the input of step 3 |

Let's look inside the VCF. The lines that start with `##` are the header; then comes a table with one row per variant. Below are the first 3 variants and the first 3 individuals.

Columns: `#CHROM` (chromosome), `POS` (position), `ID`, `REF` (reference allele), `ALT` (alternative allele), `QUAL`, `FILTER`, `INFO`, `FORMAT`, and then one column per individual: `0/0` = two reference alleles, `0/1` = one of each, `1/1` = two alternative alleles, `./.` = missing.

In [ ]:
!zcat exome.vcf.gz | grep -v '^##' | head -4 | cut -f1-12

## 3. Convert to PLINK binary files

PLINK stores each genotype in only 2 bits, so its binary files are fast to process, and ADMIXTURE reads this format directly. The conversion creates three files that always go together: `.bed` (genotypes), `.bim` (variants) and `.fam` (individuals).

In [ ]:
!./plink2 --vcf exome.vcf.gz --make-bed --out exome_data
!ls -lh exome_data.*

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `exome_data.bed` | Genotypes in binary format. It cannot be read as text: do not open it | Yes: input of step 4 |
| `exome_data.bim` | One row per variant: chromosome, variant ID, genetic position (0 = unknown), position in base pairs, allele 1, allele 2 | Yes: input of step 4 |
| `exome_data.fam` | One row per individual: family ID, individual ID, father, mother, sex (0 = unknown), phenotype (-9 = missing) | Yes: input of step 4 |
| `exome_data.log` | Record of the command and a summary of the result | No, but it is the place to check what PLINK did |

PLINK reads the three files by their common name (prefix): `--bfile exome_data` means `exome_data.bed` + `exome_data.bim` + `exome_data.fam`.

The first rows of the `.bim` (variants) and `.fam` (individuals):

In [ ]:
!head -3 exome_data.bim
!head -3 exome_data.fam

## 4. Quality control and LD pruning

We are going to use a standard quality control procedure that keeps variants:

* with a frequency of at least 1% (`--maf 0.01`)
* with a call rate of at least 90%, i.e. genotyped in at least 90% of individuals (`--geno 0.1`)
* in Hardy-Weinberg equilibrium (`--hwe 0.0001`)

And individuals:

* with at least 90% of their variants non-missing (`--mind 0.1`)

In [ ]:
!./plink2 --bfile exome_data \
       --maf 0.01 \
       --geno 0.1 \
       --mind 0.1 \
       --hwe 0.0001 \
       --make-bed \
       --out admixture_maf-0.01_geno-0.1_hwe-1e4
!ls -lh admixture_maf-0.01_geno-0.1_hwe-1e4.*
!grep -E "removed|remaining" admixture_maf-0.01_geno-0.1_hwe-1e4.log

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `admixture_maf-0.01_geno-0.1_hwe-1e4.bed / .bim / .fam` | The same data, keeping only the variants and individuals that passed the filters. The name records the filters used | Yes: input of the LD pruning (steps 4.1 and 4.2) |
| `admixture_maf-0.01_geno-0.1_hwe-1e4.log` | Summary of the filters | No. The last line of the cell shows its key lines: how many variants and individuals each filter removed |

With the course data, no individual is removed; 13,175 variants fail Hardy-Weinberg, 133,575 have a frequency below 1%, and 97,555 variants remain.

LD pruning is done in two steps:

* Windows of 50 variants, moving 5 variants at a time (`50 5`)
* In each window, we remove one variant of every pair with r² > 0.2

**Step 4.1:** generate the list of variants to keep.

In [ ]:
!./plink2 --bfile admixture_maf-0.01_geno-0.1_hwe-1e4 \
       --indep-pairwise 50 5 0.2 \
       --out ld_pruned
!wc -l ld_pruned.prune.in ld_pruned.prune.out

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `ld_pruned.prune.in` | List of the variant IDs to keep (approximately independent of each other) | Yes: in step 4.2 |
| `ld_pruned.prune.out` | List of the variant IDs removed because they are in LD with a variant that stays | No: only for checking |
| `ld_pruned.log` | Record of the command | No |

This step does not change the data: it only writes lists. With the course data, 56,823 variants stay and 40,732 go.

**Step 4.2:** keep only the variants of `ld_pruned.prune.in`.

In [ ]:
!./plink2 --bfile admixture_maf-0.01_geno-0.1_hwe-1e4 \
       --extract ld_pruned.prune.in \
       --make-bed \
       --out admixture_LD_QC
!ls -lh admixture_LD_QC.*

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `admixture_LD_QC.bed / .bim / .fam` | Final dataset: passed quality control and LD pruning (969 individuals, 56,823 variants) | Yes: input of the PCA (step 5) and of ADMIXTURE (step 7). The `.fam` also tells us who each individual is in the plots |
| `admixture_LD_QC.log` | Record of the command | No |

Why prune? PCA and ADMIXTURE assume that variants are independent. Variants in LD carry the same information and would give some regions of the genome too much weight.

## 5. Principal component analysis (PCA)

PCA summarizes the genetic differences between all individuals in a few new axes, the principal components (PCs). Individuals with similar ancestry end up close to each other. Unlike ADMIXTURE, PCA does not need us to choose a number of ancestral populations.

We use the final dataset of step 4 and compute the first 10 PCs.

In [ ]:
!./plink2 --bfile admixture_LD_QC --pca 10 --out pca
!ls -lh pca.*

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `pca.eigenvec` | One row per individual (`#FID`, `IID`) with its position on PC1 to PC10 | Yes: for the PC1 vs PC2 plot |
| `pca.eigenval` | 10 numbers, one per PC: how much variation each PC captures | Yes: for the plot of eigenvalues |
| `pca.log` | Record of the command | No |

In [ ]:
!head -3 pca.eigenvec
!cat pca.eigenval

How many PCs matter? The first PCs capture most of the population structure; after them, the values flatten out.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eigenval = pd.read_csv('pca.eigenval', header=None)[0]
ax = eigenval.plot.bar(color='#6b6b6b', figsize=(6, 3), width=0.7)
ax.set_xticklabels(range(1, len(eigenval) + 1), rotation=0)
ax.set(xlabel='Principal component', ylabel='Eigenvalue', title='Variation captured by each PC')
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

Now we read the PCs and add the population and, for the reference populations, the continent.

In [ ]:
pcs = pd.read_csv('pca.eigenvec', sep=r'\s+')
pcs['population'] = pcs['IID'].str.split('_').str[0]

continent = {'forReferenceYRI': 'African', 'forReferenceESN': 'African', 'forReferenceGWD': 'African',
             'forReferenceGBR': 'European', 'forReferenceIBS': 'European', 'forReferenceTSI': 'European',
             'forReferenceKaritiana': 'Native American', 'forReferenceSurui': 'Native American',
             'forReferencePima': 'Native American', 'forReferenceMaya': 'Native American',
             'forReferencePEL': 'Native American'}
pcs['continent'] = pcs['population'].map(continent)  # empty (NaN) for the admixed populations
pcs.head()

PC1 against PC2: the reference populations are coloured by continent, and each panel shows one admixed population in black.

In [ ]:
colors = {'African': '#eb6834', 'European': '#2a78d6', 'Native American': '#1baf7a'}
fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)

for ax, pop in zip(axes.flat, ['ACB', 'ASW', 'PUR', 'MXL']):
    for name, color in colors.items():
        ref = pcs[pcs['continent'] == name]
        ax.scatter(ref['PC1'], ref['PC2'], s=14, color=color, alpha=0.5, label=f'{name} reference')
    adm = pcs[pcs['population'] == pop]
    ax.scatter(adm['PC1'], adm['PC2'], s=14, color='black', label='Admixed population')
    ax.set_title(pop)

axes[0, 0].legend(frameon=False, fontsize=9)
for ax in axes[1]:
    ax.set_xlabel('PC1')
for ax in axes[:, 0]:
    ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

**How to read it:** each continent forms its own cluster, and admixed individuals fall *between* the clusters of their source populations; the closer an individual is to a cluster, the larger that ancestry. ADMIXTURE will put numbers on this in the next steps.

The sign of a PC is arbitrary: your plot may look mirrored compared with someone else's, and it means the same.

The plots in this step are shown here and are not saved as files.

## 6. Download ADMIXTURE

In [ ]:
!wget -nv -nc https://dalexander.github.io/admixture/binaries/admixture_linux-1.3.0.tar.gz
!tar -xzf admixture_linux-1.3.0.tar.gz
!ls -lh dist/admixture_linux-1.3.0/

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `admixture_linux-1.3.0.tar.gz` | The compressed download | No: it is only needed to extract the folder `dist/` |
| `dist/admixture_linux-1.3.0/admixture` | The ADMIXTURE program | Yes: in step 7 |
| `dist/admixture_linux-1.3.0/admixture32` | Version for old 32-bit computers | No |
| `dist/admixture_linux-1.3.0/admixture-manual.pdf` | The ADMIXTURE manual | No, but it is worth reading |
| `dist/admixture_linux-1.3.0/README.32.txt` | Notes on the 32-bit version | No |

## 7. Run ADMIXTURE

We run ADMIXTURE in unsupervised mode with K = 3, the number of ancestral populations. `-j2` uses the 2 processors that Colab provides. It takes a few minutes.

In [ ]:
!./dist/admixture_linux-1.3.0/admixture -j2 admixture_LD_QC.bed 3
!ls -lh admixture_LD_QC.3.*

**Files created in this step**

| File | What it is | Used later? |
|---|---|---|
| `admixture_LD_QC.3.Q` | Ancestry proportions: one row per individual (same order as the `.fam`), one column per ancestral component; each row adds up to 1 | Yes: for the plot in step 8, together with `admixture_LD_QC.fam` to know who each row is |
| `admixture_LD_QC.3.P` | Allele frequency of each variant in each ancestral component: one row per variant (same order as the `.bim`), one column per component | No, not in this notebook. It is used to project new individuals onto the same components |

The `3` in the names is K. ADMIXTURE does not write a log file: its messages appear only in the output of the cell.

In [ ]:
!head admixture_LD_QC.3.Q

In [ ]:
!head admixture_LD_QC.fam

## 8. Plot the ADMIXTURE results

First we read the ADMIXTURE results and the population of each individual (the first part of its ID).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Q = pd.read_csv('admixture_LD_QC.3.Q', sep=r'\s+', header=None)
fam = pd.read_csv('admixture_LD_QC.fam', sep=r'\s+', header=None)
Q['population'] = fam[1].str.split('_').str[0]
Q.head()

ADMIXTURE does not know which component is African, European or Native American: it only numbers them (0, 1, 2). We name each component after the reference population where it is highest.

In [ ]:
refs = {'forReferenceYRI': 'African', 'forReferenceGBR': 'European', 'forReferenceKaritiana': 'Native American'}
means = Q.groupby('population')[[0, 1, 2]].mean().loc[list(refs)]
means.round(2)

In [ ]:
names = {k: refs[means[k].idxmax()] for k in [0, 1, 2]}
Q = Q.rename(columns=names)
names

Now we order the individuals: reference populations first (Africa, Europe, the Americas), then the admixed populations.

In [ ]:
order = ['forReferenceYRI', 'forReferenceESN', 'forReferenceGWD',
         'forReferenceGBR', 'forReferenceIBS', 'forReferenceTSI',
         'forReferenceKaritiana', 'forReferenceSurui', 'forReferencePima', 'forReferenceMaya', 'forReferencePEL',
         'ACB', 'ASW', 'PUR', 'MXL']
Q['population'] = pd.Categorical(Q['population'], categories=order, ordered=True)
Q = Q.sort_values(['population', 'European'], ascending=[True, False]).reset_index(drop=True)

And the plot: one bar per individual, grouped by population.

In [ ]:
colors = {'African': '#eb6834', 'European': '#2a78d6', 'Native American': '#1baf7a'}
ax = Q[list(colors)].plot.bar(stacked=True, width=1, figsize=(20, 5), color=colors, edgecolor='none')

# One label per population, in the middle of its block, and a white line between blocks
size = Q.groupby('population', observed=True).size()
end = size.cumsum()
ax.set_xticks((end - size / 2 - 0.5).values)
ax.set_xticklabels(size.index.astype(str).str.replace('forReference', 'ref: '), rotation=90)
for x in end.iloc[:-1]:
    ax.axvline(x - 0.5, color='white', linewidth=1)

ax.set(ylim=(0, 1), xlabel='', ylabel='Ancestry proportion', title='ADMIXTURE, K = 3')
ax.legend(title='Ancestry', bbox_to_anchor=(1, 1), frameon=False)
for side in ax.spines.values():
    side.set_visible(False)
plt.tight_layout()
plt.show()

This step does not create files: the plot is shown here. To keep it, right-click on the image and save it, or add `plt.savefig('admixture_K3.png', dpi=300)` just before `plt.show()`.